# Synthèse comparative : PCA vs SOM

Ce notebook superpose les résultats des deux méthodes et produit les livrables comparatifs. Il est conçu pour accueillir ensuite les résultats de K-Means et de l'AutoEncoder du groupe.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import json
import plotly.graph_objects as go

## 1. Courbe Rate-Distortion superposée

In [2]:
with open("data/pca_rate_distortion.json") as f:
    pca_data = json.load(f)

with open("data/som_rate_distortion.json") as f:
    som_data = json.load(f)

fig = go.Figure()

# PCA
fig.add_trace(go.Scatter(
    x=[p["total_bytes"] for p in pca_data["points"]],
    y=[p["mse"] for p in pca_data["points"]],
    mode="lines+markers+text",
    text=[p["label"] for p in pca_data["points"]],
    textposition="top center",
    name="PCA (n_components)",
    marker=dict(size=8, symbol="circle"),
    line=dict(dash="solid")
))

# SOM
som_bytes = [p["total_bytes"] for p in som_data["points"]]
som_mse = [p["mse"] for p in som_data["points"]]
som_std = [p.get("mse_std", 0) for p in som_data["points"]]

fig.add_trace(go.Scatter(
    x=som_bytes, y=som_mse,
    error_y=dict(type="data", array=som_std, visible=True),
    mode="lines+markers+text",
    text=[p["label"] for p in som_data["points"]],
    textposition="bottom center",
    name="SOM (grid_size)",
    marker=dict(size=8, symbol="square"),
    line=dict(dash="dash")
))

fig.update_layout(
    title="Courbe Rate-Distortion : PCA vs SOM (test set, MNIST)",
    xaxis_title="Taille totale (octets) = Codebook + Latents",
    yaxis_title="MSE de reconstruction",
    xaxis_type="log",
    width=1000, height=600,
    legend=dict(x=0.7, y=0.95)
)
fig.show()

## 2. Tableau récapitulatif qualitatif

| Critère | PCA | SOM |
|---|---|---|
| **Visualisation** | Projection continue en 2D/3D (scree plot, scatter). Interprétabilité des composantes (eigendigits). Fort chevauchement inter-classes. | Carte topologique discrète (grille annotée, U-Matrix). Organisation spatiale des classes similaires. Pureté quantifiable. |
| **Compression** | Paramètre continu k, bonne granularité de la courbe R-D. Codebook = composantes + moyenne. Latent continu (float32). | Paramètre discret (taille de grille). Codebook = prototypes. Latent discret (uint8/uint16), très compact pour peu de neurones. |
| **Génération** | Limitée : décodeur linéaire + hypothèse gaussienne naïve. Images floues. Interpolation linéaire lisse mais sémantiquement pauvre. | Limitée aux K prototypes appris. Pas d'interpolation continue. Mais transitions topologiques cohérentes entre neurones voisins. |

## 3. Discussion critique

**Forces de la PCA** :
- Garanties théoriques : minimise l'erreur quadratique pour toute reconstruction linéaire de rang k
- Déterministe (pas de stochasticité)
- Coût de calcul faible (une seule SVD/eigendecomposition)
- Granularité fine du paramètre de compression (k de 1 à 784)

**Forces de la SOM** :
- Interprétabilité spatiale unique : la carte topologique permet une exploration visuelle des données
- Préservation de la topologie : classes similaires proches spatialement
- Latent discret très compact (uint8 pour grilles < 256 neurones)
- Prototypes plus interprétables que les composantes principales

**Limites partagées** :
- Aucune des deux méthodes ne modélise la vraie distribution des données
- Ni l'une ni l'autre ne peut rivaliser avec un AutoEncoder non linéaire pour la compression à très faible budget ou la génération

*A enrichir avec les résultats de K-Means et de l'AutoEncoder pour la version finale du projet.*